In [1]:
# 🛠️ ENVIRONMENT SETUP & DEPENDENCIES
# =============================================================================

print("🔧 INITIALIZING MENTAL HEALTH AGENT ENVIRONMENT")
print("=" * 70)

# Install required packages
!pip install transformers torch datasets plotly wordcloud nltk scikit-plot google-generativeai ipywidgets -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import nltk
import sys
import os
import re
import json
import warnings
import hashlib
import time
from datetime import datetime, timedelta
from collections import Counter, defaultdict
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ML & AI Imports
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from wordcloud import WordCloud

# API Imports
try:
    from kaggle_secrets import UserSecretsClient
    import google.generativeai as genai
    API_AVAILABLE = True
except ImportError:
    API_AVAILABLE = False
    print("⚠ API libraries not available - running in local mode")

warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('vader_lexicon', quiet=True)
nltk.download('stopwords', quiet=True)

print("✅ ENVIRONMENT SETUP COMPLETE")
print("📦 All dependencies loaded successfully")

🔧 INITIALIZING MENTAL HEALTH AGENT ENVIRONMENT
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 18.1 MB/s eta 0:0

2025-11-16 18:40:18.168406: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763318418.616479      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763318418.746663      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


✅ ENVIRONMENT SETUP COMPLETE
📦 All dependencies loaded successfully


In [2]:
# 🔐 API CONFIGURATION & SECURITY SETUP
# =============================================================================

print("🔐 CONFIGURING API & SECURITY SYSTEMS")
print("=" * 70)

class SecurityManager:
    def __init__(self):
        self.api_keys = {}
        self.sessions = {}
        
    def load_api_keys(self):
        """Load API keys from Kaggle secrets"""
        try:
            user_secrets = UserSecretsClient()
            self.api_keys['GOOGLE_API_KEY'] = user_secrets.get_secret("GOOGLE_API_KEY")
            if self.api_keys['GOOGLE_API_KEY']:
                print("✅ API Keys loaded from Kaggle Secrets")
                return True
            else:
                print("⚠ No API key found in Kaggle Secrets")
                return False
        except Exception as e:
            print(f"⚠ API Key Error: {str(e)}")
            print("💡 Tip: Add GOOGLE_API_KEY to Kaggle Secrets for enhanced features")
            return False
    
    def create_session(self, user_id):
        """Create user session"""
        session_id = hashlib.sha256(f"{user_id}{datetime.now()}".encode()).hexdigest()[:16]
        self.sessions[session_id] = {
            'user_id': user_id,
            'created_at': datetime.now(),
            'last_activity': datetime.now()
        }
        return session_id
    
    def validate_session(self, session_id):
        """Validate user session"""
        return session_id in self.sessions

# Initialize Security Manager
security_manager = SecurityManager()
api_available = security_manager.load_api_keys()

# Configure Gemini API if available
GEMINI_MODEL = None
if api_available and security_manager.api_keys['GOOGLE_API_KEY']:
    try:
        genai.configure(api_key=security_manager.api_keys['GOOGLE_API_KEY'])
        # Use available Gemini model
        GEMINI_MODEL = genai.GenerativeModel('models/gemini-2.0-flash')
        print("✅ Google Gemini API Configured Successfully")
        print(f"🤖 Using Model: Gemini 2.0 Flash")
    except Exception as e:
        print(f"⚠ Gemini Configuration Error: {e}")
        print("🔄 Falling back to local models...")
        GEMINI_MODEL = None
else:
    print("🔧 Running in Local Mode - Basic features available")

print("🔐 SECURITY SYSTEM INITIALIZED")

🔐 CONFIGURING API & SECURITY SYSTEMS
✅ API Keys loaded from Kaggle Secrets
✅ Google Gemini API Configured Successfully
🤖 Using Model: Gemini 2.0 Flash
🔐 SECURITY SYSTEM INITIALIZED


In [3]:
# 🛠️ TOOL FUNCTIONS & UTILITIES
# =============================================================================

print("🛠️ INITIALIZING TOOL FUNCTIONS & UTILITIES")
print("=" * 70)

class UtilityTools:
    def __init__(self):
        self.performance_metrics = {}
        
    def text_preprocessor(self, text):
        """Advanced text preprocessing"""
        if not isinstance(text, str):
            return ""
        text = re.sub(r'[^\w\s]', '', text.lower())
        text = re.sub(r'\d+', '', text)
        text = ' '.join(text.split())
        return text
    
    def calculate_similarity(self, text1, text2):
        """Calculate text similarity using TF-IDF"""
        vectorizer = TfidfVectorizer()
        try:
            tfidf_matrix = vectorizer.fit_transform([text1, text2])
            similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])
            return similarity[0][0]
        except:
            return 0.0
    
    def extract_keywords(self, text, top_n=10):
        """Extract top keywords from text"""
        words = re.findall(r'\b[a-z]{3,15}\b', text.lower())
        stop_words = set(stopwords.words('english'))
        filtered_words = [word for word in words if word not in stop_words]
        return Counter(filtered_words).most_common(top_n)
    
    def performance_tracker(self, function_name, execution_time):
        """Track function performance"""
        if function_name not in self.performance_metrics:
            self.performance_metrics[function_name] = []
        self.performance_metrics[function_name].append(execution_time)
        
    def get_performance_stats(self):
        """Get performance statistics"""
        stats = {}
        for func, times in self.performance_metrics.items():
            if times:
                stats[func] = {
                    'calls': len(times),
                    'avg_time': np.mean(times),
                    'max_time': max(times),
                    'min_time': min(times)
                }
        return stats

# Initialize Utility Tools
tools = UtilityTools()
print("✅ TOOL FUNCTIONS INITIALIZED")

🛠️ INITIALIZING TOOL FUNCTIONS & UTILITIES
✅ TOOL FUNCTIONS INITIALIZED


In [4]:
# 📝 FUNCTION DECLARATIONS & CORE LOGIC
# =============================================================================

print("📝 DECLARING CORE FUNCTIONS & BUSINESS LOGIC")
print("=" * 70)

def analyze_emotion_intensity(text):
    """Analyze emotional intensity with confidence scoring"""
    intensity_keywords = {
        'high': ['extremely', 'very', 'really', 'so', 'absolutely', 'completely'],
        'medium': ['quite', 'pretty', 'fairly', 'somewhat'],
        'low': ['slightly', 'a bit', 'a little', 'kind of']
    }
    
    text_lower = text.lower()
    intensity_score = 1.0
    
    for level, keywords in intensity_keywords.items():
        if any(keyword in text_lower for keyword in keywords):
            if level == 'high':
                intensity_score = 1.5
            elif level == 'low':
                intensity_score = 0.7
            break
    
    return intensity_score

def detect_mental_health_patterns(conversation_history):
    """Detect patterns in mental health conversations"""
    if not conversation_history:
        return {}
    
    user_messages = [entry['user'] for entry in conversation_history if 'user' in entry]
    all_text = ' '.join(user_messages).lower()
    
    patterns = {
        'symptom_frequency': {},
        'crisis_mentions': 0
    }
    
    symptoms = ['sad', 'anxious', 'stress', 'angry', 'tired', 'hopeless']
    for symptom in symptoms:
        patterns['symptom_frequency'][symptom] = all_text.count(symptom)
    
    crisis_words = ['suicide', 'kill myself', 'end it all', 'want to die']
    patterns['crisis_mentions'] = sum(all_text.count(word) for word in crisis_words)
    
    return patterns

def generate_therapeutic_response(emotion, intensity, risk_level):
    """Generate therapeutic responses based on analysis"""
    response_templates = {
        'sadness': {
            'low': "It's okay to feel sad sometimes. Consider talking to someone you trust.",
            'medium': "I hear your sadness. Remember that difficult feelings often pass with time.",
            'high': "Your sadness sounds overwhelming. Please consider reaching out for support."
        },
        'anxiety': {
            'low': "When you feel anxious, try taking a few deep breaths.",
            'medium': "Anxiety can be challenging. Grounding techniques might help you feel centered.",
            'high': "This level of anxiety sounds difficult. Professional support could provide effective strategies."
        },
        'anger': {
            'low': "Anger is a natural emotion. Try to understand what's triggering it.",
            'medium': "Your anger is valid. Finding healthy outlets can help manage these feelings.",
            'high': "Intense anger can be overwhelming. Consider speaking with a therapist."
        },
        'fear': {
            'low': "It's natural to feel fear sometimes. Try to focus on what you can control.",
            'medium': "Fear can be unsettling. Remember that you've gotten through difficult times before.",
            'high': "This level of fear sounds very challenging. Consider reaching out for support."
        }
    }
    
    if intensity > 1.2:
        intensity_level = 'high'
    elif intensity > 0.8:
        intensity_level = 'medium'
    else:
        intensity_level = 'low'
    
    emotion_lower = emotion.lower()
    if emotion_lower in response_templates:
        return response_templates[emotion_lower][intensity_level]
    else:
        return "Thank you for sharing your feelings. It's important to acknowledge and process emotions."

print("✅ CORE FUNCTIONS DECLARED")

📝 DECLARING CORE FUNCTIONS & BUSINESS LOGIC
✅ CORE FUNCTIONS DECLARED


In [5]:
# 💾 MEMORY SYSTEM & DATA MANAGEMENT
# =============================================================================

print("💾 INITIALIZING MEMORY SYSTEM & DATA MANAGEMENT")
print("=" * 70)

class MemorySystem:
    def __init__(self):
        self.conversation_memory = []
        self.user_profiles = {}
        self.max_memory_size = 1000
        
    def store_conversation(self, user_input, agent_response, analysis, session_id):
        """Store conversation with metadata"""
        memory_entry = {
            'session_id': session_id,
            'timestamp': datetime.now().isoformat(),
            'user_input': user_input,
            'agent_response': agent_response,
            'analysis': analysis,
            'emotion': analysis.get('top_emotion', {}).get('label', 'unknown'),
            'risk_level': analysis.get('risk_assessment', {}).get('level', 'unknown')
        }
        
        self.conversation_memory.append(memory_entry)
        
        if len(self.conversation_memory) > self.max_memory_size:
            self.conversation_memory.pop(0)
        
        self.update_user_profile(session_id, memory_entry)
        return memory_entry
    
    def update_user_profile(self, session_id, conversation_entry):
        """Update user profile with new conversation data"""
        if session_id not in self.user_profiles:
            self.user_profiles[session_id] = {
                'created_at': datetime.now(),
                'conversation_count': 0,
                'emotion_history': []
            }
        
        profile = self.user_profiles[session_id]
        profile['conversation_count'] += 1
        profile['emotion_history'].append(conversation_entry['emotion'])
    
    def search_conversations(self, query, session_id=None, limit=10):
        """Search through conversation history"""
        results = []
        query_lower = query.lower()
        
        for entry in reversed(self.conversation_memory):
            if session_id and entry['session_id'] != session_id:
                continue
                
            if (query_lower in entry['user_input'].lower() or 
                query_lower in entry['agent_response'].lower()):
                results.append(entry)
                
            if len(results) >= limit:
                break
        
        return results
    
    def export_data(self, session_id=None, format='json'):
        """Export conversation data"""
        if session_id:
            data = [entry for entry in self.conversation_memory if entry['session_id'] == session_id]
        else:
            data = self.conversation_memory
        
        if format == 'json':
            return json.dumps(data, indent=2, default=str)
        else:
            return "Unsupported format"
    
    def reset_memory(self, session_id=None):
        """Reset memory for specific session or all"""
        if session_id:
            self.conversation_memory = [entry for entry in self.conversation_memory 
                                      if entry['session_id'] != session_id]
            if session_id in self.user_profiles:
                del self.user_profiles[session_id]
        else:
            self.conversation_memory = []
            self.user_profiles = {}

# Initialize Memory System
memory_system = MemorySystem()
print("✅ MEMORY SYSTEM INITIALIZED")

💾 INITIALIZING MEMORY SYSTEM & DATA MANAGEMENT
✅ MEMORY SYSTEM INITIALIZED


In [6]:
# 🔐 LOGIN SYSTEM & SESSION MANAGEMENT
# =============================================================================

print("🔐 INITIALIZING LOGIN & SESSION MANAGEMENT")
print("=" * 70)

class LoginSystem:
    def __init__(self):
        self.users = {
            'demo_user': {'password': 'demo123', 'role': 'user'},
            'admin': {'password': 'admin123', 'role': 'admin'}
        }
        self.active_sessions = {}
        
    def authenticate_user(self, username, password):
        """Authenticate user credentials"""
        if username in self.users and self.users[username]['password'] == password:
            session_id = security_manager.create_session(username)
            self.active_sessions[session_id] = {
                'username': username,
                'role': self.users[username]['role'],
                'login_time': datetime.now()
            }
            return session_id
        return None
    
    def validate_session(self, session_id):
        """Validate user session"""
        return security_manager.validate_session(session_id) and session_id in self.active_sessions
    
    def logout_user(self, session_id):
        """Logout user and clear session"""
        if session_id in self.active_sessions:
            del self.active_sessions[session_id]
        return True

# Initialize Login System
login_system = LoginSystem()

# Create demo session for immediate use
DEMO_SESSION = login_system.authenticate_user('demo_user', 'demo123')
print(f"✅ LOGIN SYSTEM INITIALIZED | Demo Session: {DEMO_SESSION}")

🔐 INITIALIZING LOGIN & SESSION MANAGEMENT
✅ LOGIN SYSTEM INITIALIZED | Demo Session: 1bfb75dc7fef1aa0


In [7]:
# 🤖 MAIN MENTAL HEALTH AGENT CLASS
# =============================================================================

print("🤖 INITIALIZING MAIN MENTAL HEALTH AGENT")
print("=" * 70)

class MentalHealthAgent:
    def __init__(self):
        self.name = "MindGuard AI"
        self.version = "3.0.0"
        self.tools = tools
        self.memory = memory_system
        self.security = security_manager
        self.login = login_system
        self.gemini_model = GEMINI_MODEL
        self.api_available = GEMINI_MODEL is not None
        
        # Initialize AI models
        self.setup_ai_models()
        
        # Knowledge base
        self.setup_knowledge_base()
        
        # Performance tracking
        self.performance_stats = {
            'total_requests': 0,
            'successful_analyses': 0,
            'average_response_time': 0,
            'crisis_detections': 0,
            'api_responses': 0,
            'local_responses': 0
        }
        
        print(f"✅ {self.name} v{self.version} Initialized Successfully")
        print(f"🔗 API Status: {'ACTIVE' if self.api_available else 'LOCAL MODE'}")
    
    def setup_ai_models(self):
        """Initialize all AI models"""
        print("🔄 Loading AI Models...")
        
        try:
            # Sentiment Analysis
            self.sentiment_analyzer = pipeline("sentiment-analysis")
            
            # Emotion Detection
            self.emotion_analyzer = pipeline(
                "text-classification",
                model="j-hartmann/emotion-english-distilroberta-base",
                top_k=None
            )
            
            # Additional models
            self.sia = SentimentIntensityAnalyzer()
            
            print("✅ AI Models Loaded Successfully")
            
        except Exception as e:
            print(f"❌ Model Loading Error: {e}")
            raise
    
    def setup_knowledge_base(self):
        """Setup mental health knowledge base"""
        self.crisis_resources = {
            'immediate': [
                "National Suicide Prevention Lifeline: 1-800-273-8255",
                "Crisis Text Line: Text HOME to 741741",
                "Emergency Services: 911"
            ]
        }
        
        self.coping_strategies = {
            'anxiety': ["Practice deep breathing", "Use grounding techniques"],
            'depression': ["Behavioral activation", "Gratitude journaling"],
            'stress': ["Time management", "Mindfulness meditation"]
        }
    
    def analyze_mental_state(self, text, session_id):
        """Comprehensive mental state analysis"""
        start_time = time.time()
        
        try:
            # Multi-model analysis
            sentiment_result = self.sentiment_analyzer(text)[0]
            emotion_results = self.emotion_analyzer(text)[0]
            emotion_results_sorted = sorted(emotion_results, key=lambda x: x['score'], reverse=True)
            vader_scores = self.sia.polarity_scores(text)
            
            # Enhanced analysis
            emotion_intensity = analyze_emotion_intensity(text)
            risk_assessment = self.assess_risk_level(text)
            
            analysis_time = time.time() - start_time
            self.tools.performance_tracker('analyze_mental_state', analysis_time)
            
            analysis_result = {
                'basic_sentiment': sentiment_result,
                'emotion_breakdown': emotion_results_sorted,
                'top_emotion': emotion_results_sorted[0],
                'vader_scores': vader_scores,
                'emotion_intensity': emotion_intensity,
                'risk_assessment': risk_assessment,
                'analysis_time': analysis_time,
                'timestamp': datetime.now().isoformat()
            }
            
            self.performance_stats['successful_analyses'] += 1
            return analysis_result
            
        except Exception as e:
            print(f"❌ Analysis Error: {e}")
            return self.fallback_analysis(text)
    
    def assess_risk_level(self, text):
        """Assess mental health risk level"""
        text_lower = text.lower()
        risk_score = 0
        
        crisis_keywords = ['suicide', 'kill myself', 'end it all', 'want to die']
        for keyword in crisis_keywords:
            if keyword in text_lower:
                risk_score += 3
                self.performance_stats['crisis_detections'] += 1
        
        emotion_intensity = analyze_emotion_intensity(text)
        if emotion_intensity > 1.3:
            risk_score += 1
        
        if risk_score >= 3:
            return {'level': 'CRISIS', 'action': 'IMMEDIATE_INTERVENTION'}
        elif risk_score >= 2:
            return {'level': 'HIGH', 'action': 'PROFESSIONAL_REFERRAL'}
        elif risk_score >= 1:
            return {'level': 'MODERATE', 'action': 'MONITOR_AND_SUPPORT'}
        else:
            return {'level': 'LOW', 'action': 'SELF_CARE'}
    
    def fallback_analysis(self, text):
        """Fallback analysis when main models fail"""
        return {
            'basic_sentiment': {'label': 'NEUTRAL', 'score': 0.5},
            'top_emotion': {'label': 'neutral', 'score': 1.0},
            'risk_assessment': {'level': 'UNKNOWN', 'action': 'MONITOR'},
            'analysis_time': 0.1,
            'timestamp': datetime.now().isoformat()
        }
    
    def generate_gemini_response(self, user_input, analysis):
        """Generate response using Gemini API"""
        if not self.gemini_model:
            return None
        
        try:
            context = f"""
            User mental state analysis:
            - Primary emotion: {analysis['top_emotion']['label']}
            - Emotion intensity: {analysis['emotion_intensity']}
            - Risk level: {analysis['risk_assessment']['level']}
            - Sentiment: {analysis['basic_sentiment']['label']}
            
            User message: {user_input}
            
            Provide a compassionate, supportive mental health response.
            Focus on validation and practical coping strategies.
            Response should be 3-4 sentences maximum.
            """
            
            response = self.gemini_model.generate_content(context)
            return response.text
            
        except Exception as e:
            print(f"⚠ Gemini API failed: {e}")
            return None
    
    def generate_local_response(self, analysis):
        """Generate local response"""
        emotion = analysis['top_emotion']['label']
        intensity = analysis['emotion_intensity']
        risk_level = analysis['risk_assessment']['level']
        
        if risk_level == 'CRISIS':
            crisis_response = [
                "🚨 **URGENT SUPPORT NEEDED** 🚨",
                "",
                "Please contact:",
                "• National Suicide Prevention Lifeline: 1-800-273-8255",
                "• Crisis Text Line: Text HOME to 741741",
                "• Emergency Services: 911",
                "",
                "Your safety is the most important thing right now."
            ]
            return "\n".join(crisis_response)
        else:
            return generate_therapeutic_response(emotion, intensity, risk_level)
    
    def chat(self, user_input, session_id=DEMO_SESSION):
        """Main chat interface"""
        start_time = time.time()
        
        if not self.login.validate_session(session_id):
            return {"error": "Invalid session. Please login again."}
        
        self.performance_stats['total_requests'] += 1
        
        # Analyze user input
        analysis = self.analyze_mental_state(user_input, session_id)
        
        # Generate response - try API first, then local
        response_source = "Local AI"
        response = self.generate_gemini_response(user_input, analysis)
        
        if response:
            response_source = "Gemini API"
            self.performance_stats['api_responses'] += 1
        else:
            response = self.generate_local_response(analysis)
            response_source = "Local AI"
            self.performance_stats['local_responses'] += 1
        
        # Store conversation in memory
        memory_entry = self.memory.store_conversation(user_input, response, analysis, session_id)
        
        total_time = time.time() - start_time
        self.performance_stats['average_response_time'] = (
            self.performance_stats['average_response_time'] * (self.performance_stats['total_requests'] - 1) + total_time
        ) / self.performance_stats['total_requests']
        
        return {
            'response': response,
            'analysis': analysis,
            'memory_id': len(self.memory.conversation_memory),
            'response_time': total_time,
            'session_id': session_id,
            'response_source': response_source
        }
    
    def get_agent_stats(self):
        """Get agent performance statistics"""
        return {
            'agent_name': self.name,
            'version': self.version,
            'performance': self.performance_stats,
            'memory_usage': len(self.memory.conversation_memory),
            'active_sessions': len(self.login.active_sessions),
            'api_available': self.api_available
        }

# Initialize Main Agent
mental_health_agent = MentalHealthAgent()
print("✅ MAIN AGENT INITIALIZED SUCCESSFULLY")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


🤖 INITIALIZING MAIN MENTAL HEALTH AGENT
🔄 Loading AI Models...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cpu


✅ AI Models Loaded Successfully
✅ MindGuard AI v3.0.0 Initialized Successfully
🔗 API Status: ACTIVE
✅ MAIN AGENT INITIALIZED SUCCESSFULLY


In [8]:
# 🧪 TEST AGENT FUNCTION
# =============================================================================

print("🧪 INITIALIZING TEST AGENT FUNCTION")
print("=" * 70)

def test_agent(test_input, session_id=DEMO_SESSION, verbose=True):
    """
    Test the mental health agent with any message
    
    Usage:
    test_agent("I'm feeling exhausted")
    test_agent("I'm so anxious about everything", verbose=False)
    """
    if verbose:
        print(f"🧪 TESTING AGENT: '{test_input}'")
        print("-" * 50)
    
    start_time = time.time()
    
    try:
        # Process through agent
        result = mental_health_agent.chat(test_input, session_id)
        
        response_time = time.time() - start_time
        
        if verbose:
            print(f"💬 User Input: {test_input}")
            print(f"🤖 Agent Response: {result['response']}")
            print(f"📊 Analysis Results:")
            print(f"   🎭 Primary Emotion: {result['analysis']['top_emotion']['label']} "
                  f"(score: {result['analysis']['top_emotion']['score']:.3f})")
            print(f"   📈 Emotion Intensity: {result['analysis']['emotion_intensity']:.2f}")
            print(f"   🚨 Risk Level: {result['analysis']['risk_assessment']['level']}")
            print(f"   😊 Sentiment: {result['analysis']['basic_sentiment']['label']} "
                  f"(score: {result['analysis']['basic_sentiment']['score']:.3f})")
            print(f"   🔗 Response Source: {result['response_source']}")
            print(f"   ⏱️ Response Time: {response_time:.3f}s")
            print(f"   💾 Memory ID: {result['memory_id']}")
            print("-" * 50)
        
        return {
            'success': True,
            'input': test_input,
            'response': result['response'],
            'analysis': result['analysis'],
            'response_time': response_time,
            'memory_id': result['memory_id'],
            'response_source': result['response_source']
        }
        
    except Exception as e:
        error_msg = f"❌ Test failed: {str(e)}"
        if verbose:
            print(error_msg)
        
        return {
            'success': False,
            'input': test_input,
            'error': str(e),
            'response_time': time.time() - start_time
        }

# DEMO: Test the agent with various inputs
print("🚀 DEMONSTRATING TEST AGENT FUNCTION")
print("=" * 70)

# Test cases
test_cases = [
    "I'm feeling exhausted and overwhelmed with work",
    "I've been really anxious about my future lately",
    "I feel sad and lonely most days",
    "I'm so angry about what happened yesterday",
    "I'm feeling great and optimistic today!"
]

print("📋 RUNNING TEST CASES:")
for i, test_case in enumerate(test_cases, 1):
    print(f"   {i}. {test_case}")

print(f"\n🎯 EXECUTING {len(test_cases)} TESTS...")
print("=" * 70)

# Run all tests
test_results = []
for test_case in test_cases:
    result = test_agent(test_case, verbose=True)
    test_results.append(result)
    print()  # Add spacing between tests

# Test summary
successful_tests = sum(1 for r in test_results if r['success'])
print("📈 TEST SUMMARY:")
print(f"   ✅ Successful Tests: {successful_tests}/{len(test_cases)}")
print(f"   🤖 Agent Status: OPERATIONAL")
print(f"   🔗 API Status: {'ACTIVE' if mental_health_agent.api_available else 'LOCAL'}")

print("\n💡 TRY YOUR OWN TESTS:")
print("   test_agent('Your message here')")
print("   test_agent('I feel stressed', verbose=False)")

print("✅ TEST AGENT FUNCTION READY")

🧪 INITIALIZING TEST AGENT FUNCTION
🚀 DEMONSTRATING TEST AGENT FUNCTION
📋 RUNNING TEST CASES:
   1. I'm feeling exhausted and overwhelmed with work
   2. I've been really anxious about my future lately
   3. I feel sad and lonely most days
   4. I'm so angry about what happened yesterday
   5. I'm feeling great and optimistic today!

🎯 EXECUTING 5 TESTS...
🧪 TESTING AGENT: 'I'm feeling exhausted and overwhelmed with work'
--------------------------------------------------
💬 User Input: I'm feeling exhausted and overwhelmed with work
🤖 Agent Response: I understand you're feeling exhausted and overwhelmed with work; that sounds incredibly tough. It's okay to feel this way when facing such pressures. Let's explore some small, manageable steps you can take to regain a sense of control, like prioritizing tasks or scheduling short breaks.

📊 Analysis Results:
   🎭 Primary Emotion: surprise (score: 0.622)
   📈 Emotion Intensity: 1.00
   🚨 Risk Level: LOW
   😊 Sentiment: NEGATIVE (score: 0.997)

In [9]:
# 📊 STATISTICS DASHBOARD & ANALYTICS
# =============================================================================

print("📊 INITIALIZING STATISTICS DASHBOARD & ANALYTICS")
print("=" * 70)

class StatisticsDashboard:
    def __init__(self, agent):
        self.agent = agent
        
    def generate_comprehensive_stats(self):
        """Generate comprehensive statistics"""
        agent_stats = self.agent.get_agent_stats()
        
        memory_stats = {
            'total_conversations': len(self.agent.memory.conversation_memory),
            'active_users': len(self.agent.memory.user_profiles)
        }
        
        if self.agent.memory.conversation_memory:
            emotions = [entry['emotion'] for entry in self.agent.memory.conversation_memory]
            risk_levels = [entry['risk_level'] for entry in self.agent.memory.conversation_memory]
            
            conversation_stats = {
                'emotion_distribution': dict(Counter(emotions)),
                'risk_distribution': dict(Counter(risk_levels)),
                'most_common_emotion': max(set(emotions), key=emotions.count) if emotions else 'None'
            }
        else:
            conversation_stats = {}
        
        system_health = {
            'api_status': 'Active' if self.agent.api_available else 'Inactive',
            'models_loaded': True,
            'response_reliability': 'Excellent'
        }
        
        return {
            'agent_performance': agent_stats,
            'memory_analytics': memory_stats,
            'conversation_insights': conversation_stats,
            'system_health': system_health
        }
    
    def create_simple_dashboard(self):
        """Create a simple dashboard"""
        stats = self.generate_comprehensive_stats()
        
        # Create individual plots
        plots = []
        
        # Emotion Distribution
        if 'emotion_distribution' in stats['conversation_insights']:
            emotion_fig = px.pie(
                values=list(stats['conversation_insights']['emotion_distribution'].values()),
                names=list(stats['conversation_insights']['emotion_distribution'].keys()),
                title="🎭 Emotion Distribution",
                color_discrete_sequence=px.colors.qualitative.Set3
            )
            plots.append(emotion_fig)
        
        # Risk Distribution
        if 'risk_distribution' in stats['conversation_insights']:
            risk_fig = px.bar(
                x=list(stats['conversation_insights']['risk_distribution'].keys()),
                y=list(stats['conversation_insights']['risk_distribution'].values()),
                title="🚨 Risk Level Distribution",
                color=list(stats['conversation_insights']['risk_distribution'].keys()),
                color_discrete_map={'LOW': 'green', 'MODERATE': 'orange', 'HIGH': 'red', 'CRISIS': 'darkred'}
            )
            plots.append(risk_fig)
        
        return plots
    
    def generate_report(self):
        """Generate detailed text report"""
        stats = self.generate_comprehensive_stats()
        
        report = f"""
📊 MENTAL HEALTH AGENT - STATISTICS REPORT
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
{'=' * 60}

🤖 AGENT PERFORMANCE:
• Total Requests: {stats['agent_performance']['performance']['total_requests']}
• Successful Analyses: {stats['agent_performance']['performance']['successful_analyses']}
• Crisis Detections: {stats['agent_performance']['performance']['crisis_detections']}
• Average Response Time: {stats['agent_performance']['performance']['average_response_time']:.3f}s
• API Responses: {stats['agent_performance']['performance']['api_responses']}
• Local Responses: {stats['agent_performance']['performance']['local_responses']}

💾 MEMORY ANALYTICS:
• Total Conversations: {stats['memory_analytics']['total_conversations']}
• Active Users: {stats['memory_analytics']['active_users']}

🎭 CONVERSATION INSIGHTS:
• Most Common Emotion: {stats['conversation_insights'].get('most_common_emotion', 'N/A')}
• Emotion Distribution: {stats['conversation_insights'].get('emotion_distribution', {})}

🔧 SYSTEM HEALTH:
• API Status: {stats['system_health']['api_status']}
• Models Loaded: {stats['system_health']['models_loaded']}
• Response Reliability: {stats['system_health']['response_reliability']}
"""
        return report

# Initialize Dashboard
dashboard = StatisticsDashboard(mental_health_agent)

# Generate and display dashboard
print("📈 GENERATING DASHBOARD...")
dashboard_report = dashboard.generate_report()
print(dashboard_report)

# Show visual dashboard
print("🎨 GENERATING VISUAL DASHBOARD...")
plots = dashboard.create_simple_dashboard()
for plot in plots:
    plot.show()

print("✅ STATISTICS DASHBOARD INITIALIZED")

📊 INITIALIZING STATISTICS DASHBOARD & ANALYTICS
📈 GENERATING DASHBOARD...

📊 MENTAL HEALTH AGENT - STATISTICS REPORT
Generated: 2025-11-16 18:41:04

🤖 AGENT PERFORMANCE:
• Total Requests: 5
• Successful Analyses: 5
• Crisis Detections: 0
• Average Response Time: 1.184s
• API Responses: 5
• Local Responses: 0

💾 MEMORY ANALYTICS:
• Total Conversations: 5
• Active Users: 1

🎭 CONVERSATION INSIGHTS:
• Most Common Emotion: sadness
• Emotion Distribution: {'surprise': 1, 'fear': 1, 'sadness': 1, 'anger': 1, 'joy': 1}

🔧 SYSTEM HEALTH:
• API Status: Active
• Models Loaded: True
• Response Reliability: Excellent

🎨 GENERATING VISUAL DASHBOARD...


✅ STATISTICS DASHBOARD INITIALIZED


In [10]:
# 💾 EXPORT & RESET FUNCTIONS
# =============================================================================

print("💾 INITIALIZING EXPORT & RESET FUNCTIONS")
print("=" * 70)

class DataManager:
    def __init__(self, agent):
        self.agent = agent
        
    def export_conversations(self, session_id=None, format='json'):
        """Export conversations in various formats"""
        print(f"📤 Exporting conversations (Format: {format.upper()})...")
        
        if session_id:
            data = [entry for entry in self.agent.memory.conversation_memory 
                   if entry['session_id'] == session_id]
            filename = f"conversations_{session_id[:8]}.{format}"
        else:
            data = self.agent.memory.conversation_memory
            filename = f"all_conversations_{datetime.now().strftime('%Y%m%d_%H%M%S')}.{format}"
        
        if not data:
            return {"error": "No data available for export"}
        
        try:
            if format == 'json':
                export_data = {
                    'metadata': {
                        'export_time': datetime.now().isoformat(),
                        'total_conversations': len(data),
                        'agent_version': self.agent.version,
                        'api_used': self.agent.api_available
                    },
                    'conversations': data
                }
                content = json.dumps(export_data, indent=2, default=str)
                
            elif format == 'txt':
                content = f"Mental Health Agent Conversations\n{'='*50}\n\n"
                content += f"Exported: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
                content += f"Total Conversations: {len(data)}\n"
                content += f"API Used: {'Gemini' if self.agent.api_available else 'Local'}\n\n"
                
                for i, entry in enumerate(data, 1):
                    content += f"Conversation {i} - {entry['timestamp']}\n"
                    content += f"User: {entry['user_input']}\n"
                    content += f"Agent: {entry['agent_response']}\n"
                    content += f"Emotion: {entry.get('emotion', 'N/A')} | Risk: {entry.get('risk_level', 'N/A')}\n"
                    content += "-" * 50 + "\n\n"
            else:
                return {"error": f"Unsupported format: {format}"}
            
            with open(filename, 'w', encoding='utf-8') as f:
                f.write(content)
            
            file_size_kb = len(content) / 1024
            
            return {
                "success": True,
                "filename": filename,
                "conversations_exported": len(data),
                "file_size": f"{file_size_kb:.2f} KB"
            }
            
        except Exception as e:
            return {"error": f"Export failed: {str(e)}"}
    
    def export_analytics(self, format='json'):
        """Export system analytics"""
        print("📊 Exporting analytics data...")
        
        stats = dashboard.generate_comprehensive_stats()
        filename = f"analytics_{datetime.now().strftime('%Y%m%d_%H%M%S')}.{format}"
        
        try:
            if format == 'json':
                content = json.dumps(stats, indent=2, default=str)
            elif format == 'txt':
                content = dashboard.generate_report()
            else:
                return {"error": f"Unsupported format: {format}"}
            
            with open(filename, 'w', encoding='utf-8') as f:
                f.write(content)
            
            file_size_kb = len(content) / 1024
            
            return {
                "success": True,
                "filename": filename,
                "file_size": f"{file_size_kb:.2f} KB"
            }
            
        except Exception as e:
            return {"error": f"Analytics export failed: {str(e)}"}
    
    def reset_system(self, reset_type='conversations', session_id=None):
        """Reset system data"""
        print(f"🔄 Resetting system: {reset_type}")
        
        try:
            if reset_type == 'conversations':
                if session_id:
                    self.agent.memory.reset_memory(session_id)
                    message = f"Reset conversations for session {session_id[:8]}"
                else:
                    self.agent.memory.reset_memory()
                    message = "Reset all conversations"
                    
            elif reset_type == 'performance':
                self.agent.performance_stats = {
                    'total_requests': 0,
                    'successful_analyses': 0,
                    'average_response_time': 0,
                    'crisis_detections': 0,
                    'api_responses': 0,
                    'local_responses': 0
                }
                message = "Reset performance statistics"
                
            elif reset_type == 'all':
                self.agent.memory.reset_memory()
                self.agent.performance_stats = {
                    'total_requests': 0,
                    'successful_analyses': 0,
                    'average_response_time': 0,
                    'crisis_detections': 0,
                    'api_responses': 0,
                    'local_responses': 0
                }
                message = "Reset entire system"
                
            else:
                return {"error": f"Unknown reset type: {reset_type}"}
            
            return {
                "success": True,
                "message": message,
                "reset_type": reset_type
            }
            
        except Exception as e:
            return {"error": f"Reset failed: {str(e)}"}

# Initialize Data Manager
data_manager = DataManager(mental_health_agent)

# Test export functionality
print("🧪 TESTING EXPORT FUNCTIONALITY...")
export_result = data_manager.export_conversations(format='json')
if export_result.get('success'):
    print(f"✅ Export successful: {export_result['filename']} ({export_result['file_size']})")
else:
    print(f"❌ Export failed: {export_result.get('error', 'Unknown error')}")

print("✅ EXPORT & RESET FUNCTIONS INITIALIZED")

💾 INITIALIZING EXPORT & RESET FUNCTIONS
🧪 TESTING EXPORT FUNCTIONALITY...
📤 Exporting conversations (Format: JSON)...
✅ Export successful: all_conversations_20251116_184108.json (9.36 KB)
✅ EXPORT & RESET FUNCTIONS INITIALIZED


In [11]:
# 🔍 SEARCH FUNCTIONALITY
# =============================================================================

print("🔍 INITIALIZING SEARCH FUNCTIONALITY")
print("=" * 70)

class SearchSystem:
    def __init__(self, agent):
        self.agent = agent
        self.search_history = []
        
    def search_conversations(self, query, session_id=None, search_type='all', limit=10):
        """Advanced conversation search"""
        print(f"🔍 Searching for: '{query}' (Type: {search_type})")
        
        if not self.agent.memory.conversation_memory:
            return {"error": "No conversations available to search"}
        
        results = []
        query_lower = query.lower()
        
        for entry in reversed(self.agent.memory.conversation_memory):
            if session_id and entry['session_id'] != session_id:
                continue
            
            match_score = 0
            match_reasons = []
            
            # User input search
            if search_type in ['all', 'user'] and query_lower in entry['user_input'].lower():
                match_score += 3
                match_reasons.append("User input match")
            
            # Agent response search
            if search_type in ['all', 'agent'] and query_lower in entry['agent_response'].lower():
                match_score += 2
                match_reasons.append("Agent response match")
            
            # Emotion search
            if search_type in ['all', 'emotion'] and query_lower in entry.get('emotion', '').lower():
                match_score += 2
                match_reasons.append("Emotion match")
            
            # Risk level search
            if search_type in ['all', 'risk'] and query_lower in entry.get('risk_level', '').lower():
                match_score += 1
                match_reasons.append("Risk level match")
            
            if match_score > 0:
                results.append({
                    'entry': entry,
                    'match_score': match_score,
                    'match_reasons': match_reasons,
                    'timestamp': entry['timestamp']
                })
        
        # Sort by match score
        results.sort(key=lambda x: x['match_score'], reverse=True)
        
        # Limit results
        limited_results = results[:limit]
        
        # Store search history
        self.search_history.append({
            'query': query,
            'timestamp': datetime.now().isoformat(),
            'results_found': len(limited_results),
            'search_type': search_type
        })
        
        return {
            "query": query,
            "total_results": len(limited_results),
            "results": limited_results,
            "search_metadata": {
                "search_type": search_type,
                "session_filter": session_id,
                "limit_applied": limit
            }
        }
    
    def search_by_emotion(self, emotion, session_id=None, limit=10):
        """Search conversations by specific emotion"""
        results = []
        
        for entry in reversed(self.agent.memory.conversation_memory):
            if session_id and entry['session_id'] != session_id:
                continue
                
            if entry.get('emotion', '').lower() == emotion.lower():
                results.append(entry)
                
            if len(results) >= limit:
                break
        
        return {
            "emotion": emotion,
            "results_found": len(results),
            "conversations": results
        }
    
    def search_by_risk_level(self, risk_level, session_id=None, limit=10):
        """Search conversations by risk level"""
        results = []
        
        for entry in reversed(self.agent.memory.conversation_memory):
            if session_id and entry['session_id'] != session_id:
                continue
                
            if entry.get('risk_level', '').lower() == risk_level.lower():
                results.append(entry)
                
            if len(results) >= limit:
                break
        
        return {
            "risk_level": risk_level,
            "results_found": len(results),
            "conversations": results
        }
    
    def display_search_results(self, search_results, detailed=False):
        """Display search results in formatted way"""
        if 'error' in search_results:
            print(f"❌ Search error: {search_results['error']}")
            return
        
        print(f"🔍 SEARCH RESULTS: '{search_results['query']}'")
        print(f"📊 Found {search_results['total_results']} results")
        print("=" * 60)
        
        for i, result in enumerate(search_results['results'], 1):
            entry = result['entry']
            print(f"\n#{i} | Score: {result['match_score']:.2f}")
            print(f"🕒 {entry['timestamp']}")
            print(f"🎭 Emotion: {entry.get('emotion', 'N/A')} | 🚨 Risk: {entry.get('risk_level', 'N/A')}")
            print(f"👤 User: {entry['user_input']}")
            print(f"🤖 Agent: {entry['agent_response'][:100]}...")
            
            if detailed:
                print(f"📈 Match reasons: {', '.join(result['match_reasons'])}")
            
            print("-" * 50)

# Initialize Search System
search_system = SearchSystem(mental_health_agent)

# Test search functionality
print("🧪 TESTING SEARCH FUNCTIONALITY...")

# Test various searches
test_searches = [
    {"query": "sad", "type": "emotion"},
    {"query": "anxious", "type": "all"},
    {"query": "help", "type": "user"}
]

for search in test_searches:
    results = search_system.search_conversations(
        query=search["query"], 
        search_type=search["type"],
        limit=3
    )
    search_system.display_search_results(results)

print("✅ SEARCH FUNCTIONALITY INITIALIZED")

🔍 INITIALIZING SEARCH FUNCTIONALITY
🧪 TESTING SEARCH FUNCTIONALITY...
🔍 Searching for: 'sad' (Type: emotion)
🔍 SEARCH RESULTS: 'sad'
📊 Found 1 results

#1 | Score: 2.00
🕒 2025-11-16T18:41:01.599959
🎭 Emotion: sadness | 🚨 Risk: LOW
👤 User: I feel sad and lonely most days
🤖 Agent: I hear you, and it sounds really tough to be feeling sad and lonely so often. It's completely valid ...
--------------------------------------------------
🔍 Searching for: 'anxious' (Type: all)
🔍 SEARCH RESULTS: 'anxious'
📊 Found 1 results

#1 | Score: 5.00
🕒 2025-11-16T18:41:00.546298
🎭 Emotion: fear | 🚨 Risk: MODERATE
👤 User: I've been really anxious about my future lately
🤖 Agent: It's completely understandable to feel anxious about the future; many people experience that, especi...
--------------------------------------------------
🔍 Searching for: 'help' (Type: user)
🔍 SEARCH RESULTS: 'help'
📊 Found 0 results
✅ SEARCH FUNCTIONALITY INITIALIZED


In [12]:
# 🧪 SINGLE LINE TEST AGENT - TRY IT YOURSELF!
# =============================================================================

print("🧪 SINGLE LINE TEST AGENT - TRY IT YOURSELF!")
print("=" * 70)
print("💡 Simply edit the message in the brackets and run this cell!")
print("=" * 70)

# =============================================================================
# 🎯 EDIT THIS LINE WITH YOUR OWN MESSAGE:
# =============================================================================

test_agent("I'm feeling exhausted")

# =============================================================================
# 💡 MORE EXAMPLES TO TRY (uncomment by removing #):
# =============================================================================

# test_agent("I'm so anxious about my exam")
# test_agent("I feel really happy today!")
# test_agent("I'm angry about what happened")
# test_agent("I feel hopeless and sad")
# test_agent("I'm stressed with work deadlines")

print("\n" + "=" * 70)
print("🎯 TRY DIFFERENT MESSAGES:")
print("   • Edit the text in the brackets above")
print("   • Run this cell again")
print("   • See how the agent analyzes different emotions!")
print("=" * 70)

🧪 SINGLE LINE TEST AGENT - TRY IT YOURSELF!
💡 Simply edit the message in the brackets and run this cell!
🧪 TESTING AGENT: 'I'm feeling exhausted'
--------------------------------------------------
💬 User Input: I'm feeling exhausted
🤖 Agent Response: I hear you, it's completely understandable to feel exhausted. It's okay to acknowledge and validate that feeling. Try incorporating small breaks throughout your day, maybe a short walk or some deep breathing, to help replenish your energy. Remember, taking care of yourself is a priority.

📊 Analysis Results:
   🎭 Primary Emotion: sadness (score: 0.987)
   📈 Emotion Intensity: 1.00
   🚨 Risk Level: LOW
   😊 Sentiment: NEGATIVE (score: 1.000)
   🔗 Response Source: Gemini API
   ⏱️ Response Time: 1.089s
   💾 Memory ID: 6
--------------------------------------------------

🎯 TRY DIFFERENT MESSAGES:
   • Edit the text in the brackets above
   • Run this cell again
   • See how the agent analyzes different emotions!


In [13]:
# 🎉 FINAL COMPREHENSIVE DEMO
# =============================================================================

print("🎉 FINAL COMPREHENSIVE DEMO")
print("=" * 70)

def final_comprehensive_demo():
    """Final comprehensive demonstration of all features"""
    
    print("🚀 MENTAL HEALTH AI AGENT - COMPREHENSIVE DEMONSTRATION")
    print("=" * 70)
    
    # 1. Test agent with various emotions
    print("\n1. 🧪 TESTING EMOTION DETECTION")
    print("-" * 40)
    
    emotion_tests = [
        "I'm feeling exhausted and overwhelmed",
        "I'm really anxious about my presentation", 
        "I feel happy and content today",
        "I'm so angry about what happened",
        "I feel scared about the future"
    ]
    
    for test in emotion_tests:
        result = test_agent(test, verbose=False)
        if result['success']:
            emotion = result['analysis']['top_emotion']['label']
            risk = result['analysis']['risk_assessment']['level']
            source = result['response_source']
            print(f"✅ '{test}' → {emotion} (Risk: {risk}) | Source: {source}")
    
    # 2. Search functionality
    print("\n2. 🔍 TESTING SEARCH FUNCTIONALITY")
    print("-" * 40)
    
    search_results = search_system.search_conversations("anxious", limit=2)
    print(f"Found {search_results['total_results']} results for 'anxious'")
    
    # 3. Export data
    print("\n3. 💾 TESTING EXPORT FUNCTIONALITY")
    print("-" * 40)
    
    export_result = data_manager.export_conversations()
    if export_result.get('success'):
        print(f"✅ Exported: {export_result['filename']}")
    
    # 4. Show statistics
    print("\n4. 📊 SYSTEM STATISTICS")
    print("-" * 40)
    
    stats = mental_health_agent.get_agent_stats()
    print(f"• Agent: {stats['agent_name']} v{stats['version']}")
    print(f"• Total Requests: {stats['performance']['total_requests']}")
    print(f"• API Responses: {stats['performance']['api_responses']}")
    print(f"• Local Responses: {stats['performance']['local_responses']}")
    print(f"• Average Response Time: {stats['performance']['average_response_time']:.3f}s")
    print(f"• Memory Usage: {stats['memory_usage']} conversations")
    print(f"• API Available: {stats['api_available']}")
    
    # 5. Final message
    print(f"\n{'🎊' * 25}")
    print("🧠 MENTAL HEALTH AI AGENT - FULLY OPERATIONAL!")
    print(f"{'🎊' * 25}")
    
    print("\n🎯 ALL FEATURES WORKING:")
    print("✅ Environment Setup & API Configuration")
    print("✅ Tool Functions & Core Logic") 
    print("✅ Memory System & Data Management")
    print("✅ Login System & Security")
    print("✅ Main Agent with Emotion Detection")
    print("✅ Test Agent Function")
    print("✅ Statistics Dashboard & Analytics")
    print("✅ Export & Reset Functions")
    print("✅ Search Functionality")
    
    print("\n🚀 READY FOR USE & SUBMISSION!")
    return True

# Run the final comprehensive demo
print("🎯 STARTING FINAL COMPREHENSIVE DEMO...")
final_comprehensive_demo()

print("\n💡 QUICK START GUIDE:")
print("test_agent('your message')           - Chat with the agent")
print("search_system.search_conversations() - Search past conversations")
print("data_manager.export_conversations()  - Export data")
print("dashboard.generate_report()          - View analytics")

print("✅ FINAL COMPREHENSIVE DEMO COMPLETED!")

🎉 FINAL COMPREHENSIVE DEMO
🎯 STARTING FINAL COMPREHENSIVE DEMO...
🚀 MENTAL HEALTH AI AGENT - COMPREHENSIVE DEMONSTRATION

1. 🧪 TESTING EMOTION DETECTION
----------------------------------------
✅ 'I'm feeling exhausted and overwhelmed' → surprise (Risk: LOW) | Source: Gemini API
✅ 'I'm really anxious about my presentation' → fear (Risk: MODERATE) | Source: Gemini API
✅ 'I feel happy and content today' → joy (Risk: LOW) | Source: Gemini API
✅ 'I'm so angry about what happened' → anger (Risk: MODERATE) | Source: Gemini API
✅ 'I feel scared about the future' → fear (Risk: LOW) | Source: Gemini API

2. 🔍 TESTING SEARCH FUNCTIONALITY
----------------------------------------
🔍 Searching for: 'anxious' (Type: all)
Found 2 results for 'anxious'

3. 💾 TESTING EXPORT FUNCTIONALITY
----------------------------------------
📤 Exporting conversations (Format: JSON)...
✅ Exported: all_conversations_20251116_184115.json

4. 📊 SYSTEM STATISTICS
----------------------------------------
• Agent: MindGuar

In [14]:
test_agent("I'm feeling exhausted")

🧪 TESTING AGENT: 'I'm feeling exhausted'
--------------------------------------------------
💬 User Input: I'm feeling exhausted
🤖 Agent Response: I hear you. It's completely understandable to feel exhausted, and it's valid that you're feeling down about it. Let's focus on small steps: can you prioritize rest, even for a few minutes, and hydrate with some water? Sometimes, those simple things can help a bit.

📊 Analysis Results:
   🎭 Primary Emotion: sadness (score: 0.987)
   📈 Emotion Intensity: 1.00
   🚨 Risk Level: LOW
   😊 Sentiment: NEGATIVE (score: 1.000)
   🔗 Response Source: Gemini API
   ⏱️ Response Time: 1.050s
   💾 Memory ID: 12
--------------------------------------------------


{'success': True,
 'input': "I'm feeling exhausted",
 'response': "I hear you. It's completely understandable to feel exhausted, and it's valid that you're feeling down about it. Let's focus on small steps: can you prioritize rest, even for a few minutes, and hydrate with some water? Sometimes, those simple things can help a bit.\n",
 'analysis': {'basic_sentiment': {'label': 'NEGATIVE',
   'score': 0.9996918439865112},
  'emotion_breakdown': [{'label': 'sadness', 'score': 0.9873168468475342},
   {'label': 'neutral', 'score': 0.0041458201594650745},
   {'label': 'surprise', 'score': 0.0025306239258497953},
   {'label': 'anger', 'score': 0.0022090456914156675},
   {'label': 'disgust', 'score': 0.0015766860451549292},
   {'label': 'fear', 'score': 0.001461674110032618},
   {'label': 'joy', 'score': 0.0007593551417812705}],
  'top_emotion': {'label': 'sadness', 'score': 0.9873168468475342},
  'vader_scores': {'neg': 0.5, 'neu': 0.2, 'pos': 0.3, 'compound': -0.25},
  'emotion_intensity': 1